# Time-Series Exploratory Data Analysis (EDA) Guide

1. Overview
    - Time-series data requires a different approach to exploratory data analysis than ordinary tabular data
    - In standard tabular data
      - observations are often treated as independent
    - In time-series data
      - <b>the ordering of observations matters</b> 
      - "You" today after the "you" tmr
  
2. Objective:
    - a structured and beginner-friendly framework for performing Exploratory Data Analysis (EDA) on time-series data

3. Workflow:
    - Understand the problem → validate time → understand each series → identify temporal structure → examine relationships → test assumptions → prepare for modelling

4. Mindset:
    - how the data behaves over time and what mechanisms might explain the patterns you observe
    - for every plot, statistic, or test, ask:
      - What question am I trying to answer?
      - What pattern am I looking for?
      - What could explain that pattern?
      - Could the pattern be genuine, or could it be caused by data quality, trend, seasonality, leakage, or another statistical issue?
      - What does this imply for what I should investigate or model next?

The aim is to move from simply observing patterns to reasoning about them:

$\boxed{\text{Observe}\rightarrow\text{Question}\rightarrow\text{Hypothesise}\rightarrow\text{Test}\rightarrow\text{Interpret}\rightarrow\text{Decide}}$

---

# 1. Why Time-Series EDA Is Different

- Consider this : $Y_1,Y_2,Y_3,\ldots,Y_T$
  - In ordinary data analysis, the intuition mainly is to examine the distribution of (Y)
  - For time-series data, additionally must ask:
    - Does (Y_t) depend on (Y_{t-1})?
    - Is there a long-term trend?
    - Does the pattern repeat every day, week, month, or year?
    - Does the variance change over time?
    - Are there structural breaks or regime changes?
    - Are extreme observations errors or genuine events?
    - Are relationships between variables caused by common trends?
    - Would a predictor actually be available when making a forecast?

- Therefore:
  - $\boxed{\text{Time is part of the information contained in the data.}}$
  - ignoring the temporal structure = misleading correlations, invalid statistical inference, data leakage, and unrealistic forecasting performance

---

# 2. Overall EDA Workflow

The base workflow is (can expand on your own):

```text
START
  │
  ▼
1. DEFINE THE PROBLEM
  │
  ▼
2. VERIFY THE TIME INDEX
  │
  ▼
3. AUDIT DATA QUALITY
  │
  ▼
4. UNDERSTAND INDIVIDUAL SERIES
  │
  ├── Distribution
  ├── Missingness
  ├── Trend
  ├── Seasonality
  ├── Variance
  ├── Outliers
  └── Structural breaks
  │
  ▼
5. INVESTIGATE TEMPORAL DEPENDENCE
  │
  ├── Lag plots
  ├── ACF
  ├── PACF
  └── Periodogram / spectral analysis
  │
  ▼
6. INVESTIGATE STATIONARITY
  │
  ├── Visual diagnostics
  ├── ADF
  └── KPSS
  │
  ▼
7. INVESTIGATE VARIABLE RELATIONSHIPS
  │
  ├── Contemporaneous relationships
  ├── Lagged relationships
  ├── Spurious correlation
  └── Multicollinearity
  │
  ▼
8. PERFORM LEAKAGE AUDIT
  │
  ▼
9. CLEAN / TRANSFORM / ENGINEER FEATURES
  │
  ▼
10. CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
  │
  ▼
11. MODEL
  │
  ▼
12. RESIDUAL DIAGNOSTICS
```

---

# Step 0: Define the Problem

Before calculating anything, define what the analysis is trying to accomplish

Important questions include:

| Question                                     | Why it matters                               |
| -------------------------------------------- | -------------------------------------------- |
| What is the target (Y_t)?                    | Defines what is being predicted or explained |
| What does one observation represent?         | Determines the meaning of time               |
| What is the sampling frequency?              | Determines meaningful lags                   |
| What is the forecast horizon?                | Determines what information can be used      |
| What information exists at prediction time?  | Prevents leakage                             |
| Is the goal forecasting or causal inference? | Changes how relationships are interpreted    |
| Is the problem univariate or multivariate?   | Determines which diagnostics are relevant    |

- Example: <b><u>one-step-ahead forecasting problem:</b></u>
      - $\hat{Y}_{t+1}=f(\mathcal I_t)$
      - where $\mathcal I_t$ represents all information genuinely available at time (t)
      - IMPOT NOTE: any information from (t+1) or later must not be used

---

# Step 1: Verify the Time Index

1. Before analysing any statistical properties of the data, verify that time axis is correct
    - More specifically, check:
      * Is the timestamp stored as a datetime?
      * Are observations sorted chronologically?
      * Are timestamps duplicated?
      * Are timestamps missing?
      * Is the sampling frequency regular?
      * Are there unexpected gaps?
      * Are timezone or daylight-saving changes relevant?
    - e.g Abnormal hrly data:
      ```text
      10:00
      11:00
      13:00
      ```
      - Bro, one timestamp is missing

2. Why is matters
    - operations might refer to the **previous observation**
      ```python
      df["target"].shift(1)
      ```
    - but not necessarily the observation exactly one hour earlier when timestamps are irregular
---


# Step 2. Understand the Variables

Create a data dictionary before modelling.

For every variable identify:

* Meaning
* Unit
* Data type
* Expected range
* Whether it is continuous or categorical
* When the value becomes available
* Whether it could introduce leakage

Example:

| Variable            | Meaning                 | Type        | Availability      | Concern             |
| ------------------- | ----------------------- | ----------- | ----------------- | ------------------- |
| Target              | Variable being forecast | Continuous  | (t)               | Target              |
| Temperature         | Temperature             | Continuous  | Verify            | Potential predictor |
| Wind direction      | Direction category      | Categorical | Verify            | Encoding required   |
| Cumulative variable | Running total           | Continuous  | Verify definition | Possible leakage    |

A variable can have extremely high correlation with the target and still be useless if it would not be available when the prediction is made (AKA Spurious relationship - TERMINOLOGY ALERT)

---


# Step 3. Missing Data

- Rule 1: Do not immediately impute missing values
    - First understand the missingness

### 3.1 How much is missing?

- e.g variable $X_j$:
    - $MissingRate_j=\frac{\text{Number of missing observations}}{\text{Total observations}}$

### 3.2 When is data missing?

- consider plotting missing observations over time
- Look for:
    - isolated missing observations
    - consecutive gaps
    - long missing periods
    - missingness concentrated in certain seasons
    - missingness during extreme events

### 3.3 How long are the gaps?

Guidelines (not set and stone):
1. single missing observation and missing months worth of observations should not automatically receive the same treatment
2. Interpolation across a short gap may be reasonable
3. Interpolation across a long gap may manufacture data that never existed
4. Also note distinguish between:
    - **Missing value:** the timestamp exists but the value is `NaN`
    - **Missing timestamp:** the entire observation is absent

---

# Step 4. Univariate Distribution

Aim: understand the values taken by each variable, before examining temporal behaviour

1. Useful statistics include:
    - Count, Mean, Median, Standard deviation, Minimum, Maximum, Quantiles, Skewness

2. Useful visualisations include:
    - Histogram, Boxplot, Density plot

3. Note that:

$\boxed{\text{Distribution does not describe temporal structure.}}$

4. Reason:
    - <b><u>Two time series can contain exactly the same values but appear in completely different orders</b></u>
    - Meaning (for example)
        - histograms answer: **What values occur?**
        - Time-series plots answer: **When and in what sequence do they occur?**
    - Both are required

---

# Step 5. Plot the Series Across Multiple Time Scales

- Use multiple resolutions:
     - FULL HISTORY $\rightarrow$ YEAR / QUARTER $\rightarrow$ MONTH / WEEK $\rightarrow$ DAY / INTRADAY

1. Full history
     - Look for:
          * Long-term trend
          * Regime changes
          * Structural breaks
          * Changing variance
          * Extreme periods

2. Intermediate windows
     - Look for:
          * Seasonal behaviour
          * Medium-term cycles
          * Volatility clusters

3. Short windows
     - Look for:
          * Local dynamics
          * Persistence
          * Sudden spikes
          * Intraday patterns

TLDR: Different temporal resolutions reveal different structures

---

# 9. Trend

A **trend** is a persistent long-run movement in the level of a time series.

Possible tools include:

* Rolling mean
* Rolling median
* Resampling
* Smoothing
* Decomposition

For a rolling mean:

$
MA_t(w)=\frac{1}{w}\sum_{i=0}^{w-1}Y_{t-i}
$

The window (w) should have a meaningful interpretation.

For hourly data:

$
w=24
$

represents approximately one day.

$
w=168
$

represents approximately one week.

Look for whether the expected level:

$
E(Y_t)
$

appears to change systematically over time.

---|





# 10. Seasonality

**Seasonality** is systematic behaviour that repeats at predictable intervals.

For hourly data, investigate patterns by:

* Hour
* Day of week
* Week
* Month
* Year
* Holiday or working day where relevant

For example:

$
E(Y_t\mid Hour=h)
$

shows the average behaviour at a particular hour.

Useful visualisations include:

* Average by hour
* Average by weekday
* Average by month
* Boxplots by seasonal category
* Year-over-year plots
* Hour × weekday heatmaps

Repeating patterns suggest that seasonal information may need to be represented explicitly in the model.

---

# 11. Time-Series Decomposition

A time series can sometimes be represented as:

$
Y_t=T_t+S_t+R_t
$

where:

* (T_t) = trend
* (S_t) = seasonal component
* (R_t) = remainder

The purpose of decomposition is to ask:

> **After removing systematic trend and seasonality, what structure remains?**

Common approaches include:

* Classical seasonal decomposition
* STL decomposition

For multiplicative behaviour:

$
Y_t=T_tS_tR_t
$

A logarithmic transformation can sometimes convert multiplicative relationships into approximately additive ones.

---

# 12. Variance Stability

A covariance-stationary process requires its unconditional variance to remain constant:

$
Var(Y_t)=\sigma^2
$

Useful diagnostics include:

* Rolling standard deviation
* Rolling variance
* Variance by year/month
* Mean versus variance relationships

Look for periods where fluctuations become systematically larger or smaller.

Possible responses include:

* Log transformation
* Box-Cox transformation
* Explicit variance modelling

However, changing variance in the **raw series** should not be confused with heteroskedasticity in regression residuals.

They are related diagnostic concerns, but they are not the same assumption.

---

# 13. Outliers

An extreme observation is not automatically an error.

For time-series data, distinguish between:

1. Data error
2. Genuine extreme event
3. Global outlier
4. Local outlier

A useful investigation process is:

```text
Extreme observation
        │
        ▼
Physically impossible?
   │            │
  YES           NO
   │            │
Possible      Check surrounding
error         observations
                │
                ▼
         Supported by other
         variables/events?
          │           │
         YES          NO
          │           │
       Genuine     Investigate
        event       further
```

Useful methods include:

* IQR
* Robust z-scores
* Median Absolute Deviation (MAD)
* Rolling/local deviations
* Domain-specific limits

Never automatically delete extreme observations simply because a statistical rule labels them as outliers.

---

# 14. Structural Breaks and Regime Changes

A structural break occurs when the data-generating process changes.

For example:

$Y_t=\begin{cases}\mu_1+u_t,&t<T^*\\\mu_2+u_t,&t\geq T^*\end{cases}$

Possible changes include:

* Mean
* Variance
* Trend
* Seasonal pattern
* Relationship between variables

Look for structural changes using:

* Time plots
* Rolling statistics
* Comparisons across periods
* Change-point methods
* Formal structural-break tests where appropriate

Historical observations from a previous regime may be less informative about future behaviour.

---

# 15. Autocorrelation

Autocorrelation measures the relationship between a variable and its own lagged values.

At lag (k):

$
\rho_k=Corr(Y_t,Y_{t-k})
$

## Lag plots

Begin with intuitive comparisons such as:

$
Y_t \text{ vs } Y_{t-1}
$

or:

$
Y_t \text{ vs } Y_{t-24}
$

for hourly data.

## Autocorrelation Function (ACF)

The ACF examines correlations across many lags:

$
\rho_1,\rho_2,\ldots,\rho_k
$

Look for:

* Strong short-lag correlations
* Slow decay
* Rapid decay
* Repeating seasonal peaks

For hourly data, peaks around:

$
24,48,72,\ldots
$

may indicate daily seasonality.

---

# 16. Partial Autocorrelation Function (PACF)

The PACF measures the relationship between (Y_t) and (Y_{t-k}) after controlling for intermediate lags.

For example, (Y_t) may appear related to (Y_{t-2}) simply because:

$
Y_{t-2}\rightarrow Y_{t-1}\rightarrow Y_t
$

PACF helps identify whether lag 2 contributes additional information after accounting for lag 1.

This is useful for:

* Understanding temporal structure
* Selecting lagged features
* Developing intuition for autoregressive models

---

# 17. Frequency-Domain Analysis

Time-series patterns can also be studied in terms of repeating frequencies.

The relationship between frequency and period is:

$
f=\frac{1}{P}
$

where (P) is the period.

For hourly observations:

$
P=24
$

represents a daily cycle.

The two perspectives complement each other:

```text
TIME DOMAIN              FREQUENCY DOMAIN

ACF                      Periodogram
 │                            │
 ▼                            ▼
Which lags matter?       Which cycles matter?
```

Periodograms can help identify dominant periodic behaviour that may not be obvious from the raw time plot.

---

# 18. Stationarity

A covariance-stationary series has:

$
E(Y_t)=\mu
$

$
Var(Y_t)=\sigma^2
$

and:

$
Cov(Y_t,Y_{t-k})
$

depends on the lag (k), rather than the specific calendar time (t).

Informally:

> The statistical behaviour of the process remains broadly stable over time.

## Visual evidence

Look for:

* Trend
* Changing mean
* Changing variance
* Persistent autocorrelation
* Structural breaks
* Changing seasonality

## Augmented Dickey-Fuller (ADF)

Broadly:

$
H_0:\text{unit root}
$

Failure to reject the null can indicate evidence consistent with nonstationarity.

## KPSS

Broadly:

$
H_0:\text{stationarity}
$

Using ADF and KPSS together can provide complementary evidence.

| ADF            | KPSS           | Possible interpretation                    |
| -------------- | -------------- | ------------------------------------------ |
| Reject         | Fail to reject | Evidence supporting stationarity           |
| Fail to reject | Reject         | Evidence supporting nonstationarity        |
| Reject         | Reject         | Conflicting evidence / investigate further |
| Fail to reject | Fail to reject | Inconclusive                               |

Statistical tests should support reasoning rather than replace it.

---

# 19. Transformations and Differencing

If the series appears nonstationary, first identify the likely mechanism.

```text
Nonstationarity
      │
      ├── Trend
      ├── Seasonality
      ├── Changing variance
      └── Structural break
             │
             ▼
      Appropriate transformation
             │
             ▼
        Repeat diagnostics
```

## First difference

$
\Delta Y_t=Y_t-Y_{t-1}
$

may help remove stochastic trends.

## Seasonal difference

For hourly data with daily seasonality:

$
\Delta_{24}Y_t=Y_t-Y_{t-24}
$

## Log transformation

$
Z_t=\log(Y_t)
$

may help when variability increases with the level of the series.

After transformation, repeat relevant plots, ACF/PACF, and stationarity diagnostics.

Do not mechanically difference a series until a statistical test produces the desired p-value.

---

# 20. Relationships Between Variables

Once individual variables are understood, investigate their relationships.

For (X_t) and (Y_t), consider:

* Scatterplots
* Pearson correlation
* Spearman correlation
* Conditional plots
* Categorical comparisons

However:

$
Corr(X_t,Y_t)\neq\text{causation}
$

Time series introduce another major problem: two variables can appear highly correlated because they share a common trend.

---

# 21. Spurious Correlation

Suppose:

$
X_t=t+\epsilon_t
$

and:

$
Y_t=2t+\eta_t
$

Even when (\epsilon_t) and (\eta_t) are unrelated, (X_t) and (Y_t) may have a very high correlation because both trend over time.

One useful diagnostic is to compare:

$
Corr(X_t,Y_t)
$

with:

$
Corr(\Delta X_t,\Delta Y_t)
$

A large reduction after differencing suggests that common temporal structure may have contributed to the original relationship.

Differencing does **not prove** that a relationship is spurious. It is one diagnostic among several.

---

# 22. Lagged Relationships

Forecasting often cares more about:

$
Corr(X_{t-k},Y_t)
$

than:

$
Corr(X_t,Y_t)
$

because past information may help predict future values.

Useful tools include:

* Lagged scatterplots
* Lagged correlations
* Cross-Correlation Function (CCF)

Potential features might therefore include:

$
X_{t-1},X_{t-2},X_{t-24},X_{t-168}
$

depending on the data frequency and domain.

Interpret cross-correlations carefully because trend, seasonality, and autocorrelation can create misleading relationships.

---

# 23. Multicollinearity

The time-series OLS assumption of no perfect collinearity requires that one explanatory variable cannot be reconstructed exactly from the others.

Perfect multicollinearity prevents unique estimation of regression coefficients.

Useful diagnostics include:

* Correlation matrix
* Scatterplots
* Design-matrix rank
* Variance Inflation Factor (VIF)

Distinguish between:

**Perfect multicollinearity:** estimation is impossible without removing redundant information.

**High multicollinearity:** estimation may remain possible, but coefficient estimates can become unstable and imprecise.

---

# 24. Data Leakage

Before modelling, perform an explicit leakage audit.

For every predictor ask:

$
\boxed{\text{Would this information genuinely exist when the prediction is made?}}
$

Common sources of time-series leakage include:

* Future observations
* Centered rolling averages
* Backward filling
* Target-derived future features
* Future aggregates
* Scaling using the entire dataset
* Feature selection using test data
* Interpolation using future information
* Random train/test splitting

For example:

$
MA_t=
\frac{Y_{t-1}+Y_t+Y_{t+1}}{3}
$

contains (Y_{t+1}).

If forecasting at time (t), this feature contains future information and therefore creates leakage.

---

# 25. Chronological Train/Test Splitting

Time-series data should normally preserve chronological order.

Instead of randomly splitting:

```text
Random observations → Train/Test
```

prefer:

```text
PAST                     FUTURE

Train → Validation → Test
```

The model should be evaluated on data occurring **after** the data used for training.

This better approximates the real forecasting problem:

$
\text{Learn from past} \rightarrow \text{Predict future}
$

Walk-forward or rolling validation may be appropriate when more robust evaluation is required.

---

# 26. EDA Versus Model Diagnostics

A critical distinction:

$
\boxed{\text{EDA diagnoses the data-generating process}}
$

while:

$
\boxed{\text{Residual diagnostics diagnose the fitted model}}
$

Suppose:

$
Y_t=X_t'\beta+u_t
$

After estimating the model:

$
\hat u_t=Y_t-\hat Y_t
$

The assumptions concerning (u_t) should then be investigated using the **residuals**.

A raw target can be strongly autocorrelated while a correctly specified model produces approximately uncorrelated residuals.

---

# 27. Connection to Time-Series OLS Assumptions

The EDA framework can be connected to the common time-series OLS assumptions
(But moreso if ur goal is statistical inference or causal interpretation)
(if ur goal is pure prediction, then this part no matter)

## TS1': Linearity and weak dependence

Investigate whether relationships can reasonably be represented using the proposed functional form and whether temporal dependence behaves appropriately.

Relevant diagnostics include:

* Time plots
* Scatterplots
* Transformations
* ACF/PACF
* Stationarity analysis

Weak dependence and covariance stationarity are related concepts, but they should not be treated as identical.

---

## TS2': No Perfect Collinearity

Ask:

> Can one predictor be reconstructed exactly from the others?

Relevant diagnostics:

* Correlation matrix
* Matrix rank
* VIF for high multicollinearity
* Feature-definition review

---

## TS3': Zero Conditional Mean

Conceptually:

$
E(u_t\mid X)=0
$

Potential violations include:

* Omitted variables
* Simultaneity
* Measurement error
* Feedback relationships
* Incorrect dynamic specification
* Endogeneity

EDA can reveal warning signs but **cannot prove zero conditional mean**.

Exogeneity is fundamentally a modelling and identification assumption.

---

## TS4': Contemporaneous Homoskedasticity

For the regression error:

$
Var(u_t\mid X_t)=\sigma^2
$

After fitting a model, investigate:

* Residuals vs fitted values
* Residuals over time
* Rolling residual variance
* Breusch-Pagan test
* White test where appropriate

Changing variance in the raw target is not itself proof that TS4' is violated.

---

## TS5': No Serial Correlation

For regression errors:

$
Corr(u_t,u_{t-k}\mid X)=0
$

After fitting the model, investigate:

* Residual time plot
* Residual ACF
* Ljung-Box test
* Breusch-Godfrey test

Significant residual autocorrelation suggests that predictable temporal information may remain unexplained by the model.

Possible missing structure includes:

* Lagged target values
* Lagged predictors
* Trend
* Seasonality
* Structural changes

---

# 28. Standard Structure for Every EDA Check

Each diagnostic in this notebook should answer five questions:

### 1. Concept

**What does the term mean?**

Define the statistical idea clearly.

### 2. Why

**Why do we care?**

Explain what can go wrong if the property is ignored.

### 3. Method

**How do we investigate it?**

Introduce the relevant plots, statistics, or tests.

### 4. Look For

**What patterns indicate a potential problem?**

Interpret the output rather than simply generating it.

### 5. Action

**What should we do next?**

EDA should lead to a modelling, cleaning, or investigation decision.

The objective is therefore:

$
\boxed{
\text{Question}
\rightarrow
\text{Diagnostic}
\rightarrow
\text{Interpretation}
\rightarrow
\text{Decision}
}
$

---

# 29. Quick Reference Checklist

Before modelling a time series, verify:

* $ $ Problem and forecast horizon are clearly defined
* $ $ Datetime index is correct
* $ $ Observations are chronologically sorted
* $ $ Duplicate timestamps are checked
* $ $ Missing timestamps are identified
* $ $ Missing values and gap lengths are understood
* $ $ Variable definitions and units are understood
* $ $ Feature availability has been verified
* $ $ Distributions have been inspected
* $ $ Full-history time plots have been inspected
* $ $ Multiple time resolutions have been examined
* $ $ Trend has been investigated
* $ $ Seasonality has been investigated
* $ $ Variance stability has been investigated
* $ $ Outliers have been investigated rather than automatically removed
* $ $ Structural breaks have been considered
* $ $ Lag plots have been examined
* $ $ ACF has been examined
* $ $ PACF has been examined where relevant
* $ $ Periodicity has been investigated
* $ $ Stationarity has been assessed
* $ $ ADF/KPSS have been interpreted alongside visual evidence
* $ $ Contemporaneous relationships have been examined
* $ $ Spurious correlation has been considered
* $ $ Lagged relationships have been examined
* $ $ Multicollinearity has been investigated
* $ $ Data leakage has been explicitly audited
* $ $ Transformations are justified by a statistical reason
* $ $ Train/validation/test splits preserve chronology
* $ $ Preprocessing is fitted using training information only
* $ $ Residual diagnostics are performed after modelling

---

# Final Principle

Time-series EDA is not a checklist of plots to generate mechanically.

Every diagnostic should answer a question about the process generating the observations.

The central workflow is:

$
\boxed{
\text{Understand}
\rightarrow
\text{Diagnose}
\rightarrow
\text{Explain}
\rightarrow
\text{Transform}
\rightarrow
\text{Model}
\rightarrow
\text{Validate}
}
$

A good time-series analysis should be able to explain not only **what the data looks like**, but also:

> **Why does it look this way, what statistical mechanism could produce the pattern, what modelling assumption could it affect, and what should we do about it?**

That reasoning process is the real purpose of exploratory data analysis.
